# HUG-IML Classifier — Pure Python Sample Notebook

This notebook demonstrates the **pure-Python backend** (`HUGIMLClassifierPy`) of the HUG-IML classifier described in:

> Krishnamoorthy, S. (2024). *Interpretable classifier models for decision support using high utility gain patterns.* IEEE Access, 12, 126088–126107. https://doi.org/10.1109/ACCESS.2024.3455563

Unlike `HUGIMLClassifier`, this backend requires **no Java runtime** and no intermediate disk files. All processing is done in pure Python using NumPy, pandas, SciPy, and scikit-learn.

**Dataset used:** Pima Indians Diabetes (National Institute of Diabetes and Digestive and Kidney Diseases)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd

from HUGIMLClassifierPy import HUGIMLClassifierPy
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

## 2. Load Dataset

In [ ]:
fname = 'datasets/pima indians diabetes.csv'
data = pd.read_csv(fname, header=None)
data.columns = [
    'numPregnancies', 'glucose', 'bp', 'skinThickness',
    'insulin', 'bmi', 'diabetesPedigreeFunction', 'age', 'class'
]

X = data.iloc[:, :-1]
y = data.iloc[:, -1]

print('Dataset shape:', X.shape)
print('Class distribution:')
print(y.value_counts())
X.head()

## 3. Identify Column Types and Set Parameters

`HUGIMLClassifierPy` requires three lists grouped by column type:
- **Integer columns** — binned on raw integer values
- **Float columns** — MinMax-scaled before binning
- **Categorical columns** — treated as one item per unique value

Key parameters:
| Parameter | Value | Description |
|-----------|-------|-------------|
| `B` | `-1` | Auto-select bin count (maximises per-column information gain over [2, 20]) |
| `L` | `1` | Mine singleton patterns only |
| `G` | `1e-6` | Minimum information-gain threshold for a pattern to be retained |

In [ ]:
numericIntCols   = [c for c in X.columns if np.issubdtype(X[c].dtype, np.integer)]
numericFloatCols = [c for c in X.columns if np.issubdtype(X[c].dtype, float)]
catCols          = [c for c in X.columns if np.issubdtype(X[c].dtype, object)]
allCols          = [numericIntCols, numericFloatCols, catCols]

print('Integer columns :', numericIntCols)
print('Float columns   :', numericFloatCols)
print('Categorical cols:', catCols)

params = {
    'B'          : -1,
    'L'          : 1,
    'G'          : 1e-6,
    'allCols'    : allCols,
    'origColumns': X.columns.tolist()
}

## 4. Train / Test Split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

print(f'Training samples : {len(x_train)}')
print(f'Test samples     : {len(x_test)}')

## 5. Fit the Classifier

In [ ]:
clf = HUGIMLClassifierPy(**params)
clf.fit(x_train, y_train)

print('Training pattern matrix shape:', clf.get_transformed_shape())
print('Number of HUG patterns mined :', len(clf.get_hug_features()))

## 6. Predict and Evaluate

In [ ]:
y_pred_proba = clf.predict_proba(x_test)
y_prob       = y_pred_proba[:, 1]
y_pred       = np.argmax(y_pred_proba, axis=1)

acc  = accuracy_score(y_test, y_pred)
bacc = balanced_accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print('================ TEST METRICS ================')
print(f'Accuracy           : {acc:.4f}')
print(f'Balanced Accuracy  : {bacc:.4f}')
print(f'Precision          : {prec:.4f}')
print(f'Recall             : {rec:.4f}')
print(f'F1 Score           : {f1:.4f}')
print(f'ROC AUC            : {auc:.4f}')

In [ ]:
print('Confusion Matrix')
print(confusion_matrix(y_test, y_pred))

print('\nClassification Report')
print(classification_report(y_test, y_pred, digits=4))

## 7. Inspect HUG Patterns

`get_hug_features()` returns human-readable pattern labels.  
`get_pattern_info()` returns a DataFrame with utility, information gain, and support per pattern.

In [ ]:
print('HUG Pattern labels:')
for p in clf.get_hug_features():
    print(' ', p)

In [ ]:
pattern_df = clf.get_pattern_info()
pattern_df.sort_values('utility', ascending=False)